# ROCLING 2026 DSA — E10 / E11 / E12（multi-seed + 多 encoder + ensemble）

專跑 **拿分主線**，不需要 L2/L3 的大 pkl。流程：
1. E10：MacBERT + L1 跑 5 個 seed
2. E11：RoBERTa-wwm-ext 與 large 各一顆
3. E12：把所有 run 用 dimension-wise weighted / mean 融合出 submission

**執行前先把 Runtime 改成 GPU**（選單 → 執行階段 → 變更執行階段類型 → T4 GPU）。
每顆模型 T4 約 8–15 分鐘；E10 五顆 + E11 兩顆 ≈ 1–1.5 小時。

In [ ]:
# 1) 安裝套件
!pip -q install "transformers>=4.40" jieba scikit-learn scipy
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!')

In [ ]:
# 2) 上傳 baseline_ens.zip 並解壓、切到含 train_v2.py 的目錄
from google.colab import files
up = files.upload()              # 選 baseline_ens.zip
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z: z.extractall('.')
for root, _, fs in os.walk('.'):
    if 'train_v2.py' in fs and 'data' in os.listdir(root):
        os.chdir(root); break
os.makedirs('outputs/preds', exist_ok=True)
print('工作目錄:', os.getcwd()); print(sorted(os.listdir('.')))

## E10 — Multi-seed ensemble（MacBERT + L1 × 5 seeds）
兩篇得獎論文共識的最大槓桿：同架構跑多個 seed，平掉方差、穩定 arousal 排序。

In [ ]:
# 3) E10：5 個 seed（每顆存 outputs/macbert_s{seed}_best.pt 與 preds/）
for s in [42, 1, 2, 3, 4]:
    print(f'\n===== seed {s} =====')
    !python train_v2.py --seed {s} --run_name macbert_s{s} --epochs 4 --batch_size 32

In [ ]:
# 4) E10 融合（等權平均；multi-seed 用 mean 最穩，不易 overfit 253 筆 dev）
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    --mode mean --name e10_seed_ens

## E11 — RoBERTa-wwm-ext + L1
換 encoder bias。large 版對長反思文本可能更會排 arousal，但 T4 記憶體吃緊 → **batch 8 + lr 1e-5**。

In [ ]:
# 5) E11a：RoBERTa-wwm-ext（base）
!python train_v2.py --model hfl/chinese-roberta-wwm-ext --run_name roberta_s42 --epochs 4 --batch_size 32

In [ ]:
# 6) E11b：RoBERTa-wwm-ext-large（T4 一定要 batch 8 + lr 1e-5，否則 OOM）
!python train_v2.py --model hfl/chinese-roberta-wwm-ext-large \
    --run_name robertaL_s42 --epochs 4 --batch_size 8 --lr 1e-5

## E12 — 跨 encoder dimension-wise weighted ensemble
valence / arousal **分開**算權重（`score = max(PCC,0)/(MAE+eps)`），arousal 維度只讓會排 arousal 的模型投票。

> dev 只有 253 筆，weighted 可能 overfit → 下面同時跑 mean 版比較；兩者 dev 差距大時**選保守的 mean**。

In [ ]:
# 7) E12：把手上所有 run 丟進去（weighted）
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    roberta_s42 robertaL_s42 --mode weighted --name e12_enc_ens

print('\n----- 對照：mean 版 -----')
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    roberta_s42 robertaL_s42 --mode mean --name e12_enc_mean

## 下載結果
`*_submission.csv` 是官方格式（ID,Valence,Arousal）。`outputs/preds/` 裡的統一預測檔留著，
之後要做 E13 偽標（teacher 用這些 `*_best.pt`）或 E17 校準都會用到。

In [ ]:
# 8) 打包下載所有 submission + 預測檔 + 權重
import shutil, os
os.makedirs('to_download', exist_ok=True)
for f in os.listdir('outputs'):
    if f.endswith('_submission.csv') or f.endswith('_best.pt'):
        shutil.copy(f'outputs/{f}', 'to_download/')
shutil.make_archive('e10_e12_results', 'zip', 'outputs')   # 含 preds/ 與所有權重
from google.colab import files
files.download('e10_e12_results.zip')
print('已打包 outputs/（submission + preds + best_model 權重）')

In [ ]:
# 9) （可選）只看各 submission 的預測分布，快速 sanity check
import pandas as pd, glob
for p in sorted(glob.glob('outputs/*_submission.csv')):
    df = pd.read_csv(p)
    print(f"{os.path.basename(p):32s} V {df.Valence.mean():.2f}±{df.Valence.std():.2f}  "
          f"A {df.Arousal.mean():.2f}±{df.Arousal.std():.2f}")